# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [1]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = './data'

In [3]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [4]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [5]:
char_to_id = {}
id_to_char = {}

# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to convert characters to ids, while id_to_char is the opposite

# Add special tokens first
char_to_id['<pad>'] = 0    # Padding token for batch processing
char_to_id['<eos>'] = 1    # End of sequence token
char_to_id['<unk>'] = 2    # Unknown character (for unseen chars in eval set)

id_to_char[0] = '<pad>'
id_to_char[1] = '<eos>'
id_to_char[2] = '<unk>'

# Get unique characters from training data
idx = 3
for char in sorted(set(''.join(df_train['src'].unique()))):
    if char not in char_to_id:
        char_to_id[char] = idx
        id_to_char[idx] = char
        idx += 1

vocab_size = len(char_to_id)
print('Vocab size: {}'.format(vocab_size))
print('Character mappings (first 20):')
for i in range(min(20, vocab_size)):
    print(f'  {id_to_char[i]}: {i}')


Vocab size: 19
Character mappings (first 20):
  <pad>: 0
  <eos>: 1
  <unk>: 2
  (: 3
  ): 4
  *: 5
  +: 6
  -: 7
  0: 8
  1: 9
  2: 10
  3: 11
  4: 12
  5: 13
  6: 14
  7: 15
  8: 16
  9: 17
  =: 18


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [7]:
# Data Preprocessing
# Convert each character to ID and append <eos> token
# Example: "1+2=" -> [9, 6, 10, 18, 1] (where 1 is <eos>)

def char_to_id_with_unk(char):
    """Convert character to ID, use <unk> for unknown characters"""
    return char_to_id.get(char, char_to_id['<unk>'])

# Process training data
df_train['char_id_list'] = df_train['src'].apply(
    lambda x: [char_to_id_with_unk(c) for c in x] + [char_to_id['<eos>']]
)

# For labels, we need to handle the output format:
# Before '=': pad tokens (represented as position markers)
# After '=': target answer + <eos>
def create_label(row):
    """
    Create label sequence:
    - Before and including '=': use <pad> token (no need to predict)
    - After '=': the target answer character + <eos>
    
    For example: "1+2-3=" with target "0"
    Label: [0, 0, 0, 0, 0, 0, char_to_id['0'], char_to_id['<eos>']]
    (6 pads for "1+2-3=" + answer '0' + <eos>)
    """
    src = row['src']
    tgt = row['tgt']
    
    # Find the position of '=' in source
    eq_pos = src.index('=')
    # Create label: pad for input part, then answer, then <eos>
    label = [char_to_id['<pad>']] * (eq_pos + 1) + \
            [char_to_id_with_unk(c) for c in str(tgt)] + \
            [char_to_id['<eos>']]
    
    return label

df_train['label_id_list'] = df_train.apply(create_label, axis=1)

# 顯示處理前後的對比
print("=== Data Preprocessing Examples ===\n")
for idx in range(min(3, len(df_train))):
    row = df_train.iloc[idx]
    src = row['src']
    tgt = row['tgt']
    char_ids = row['char_id_list']
    label_ids = row['label_id_list']
    
    print(f"Example {idx + 1}:")
    print(f"  Source: {src}")
    print(f"  Target: {tgt}")
    print(f"  Char IDs: {char_ids}")
    print(f"  Label IDs: {label_ids}")
    print(f"  Lengths: char={len(char_ids)}, label={len(label_ids)}\n")

# Show statistics
print("=== Dataset Statistics ===")
print(f"Total samples: {len(df_train)}")
print(f"Average char sequence length: {df_train['char_id_list'].apply(len).mean():.2f}")
print(f"Max char sequence length: {df_train['char_id_list'].apply(len).max()}")
print(f"Min char sequence length: {df_train['char_id_list'].apply(len).min()}")


=== Data Preprocessing Examples ===

Example 1:
  Source: 14*(43+20)=882
  Target: 882
  Char IDs: [9, 12, 5, 3, 12, 11, 6, 10, 8, 4, 18, 16, 16, 10, 1]
  Label IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 16, 16, 10, 1]
  Lengths: char=15, label=15

Example 2:
  Source: (6+1)*5=35
  Target: 35
  Char IDs: [3, 14, 6, 9, 4, 5, 13, 18, 11, 13, 1]
  Label IDs: [0, 0, 0, 0, 0, 0, 0, 0, 11, 13, 1]
  Lengths: char=11, label=11

Example 3:
  Source: 13+32+29=74
  Target: 74
  Char IDs: [9, 11, 6, 11, 10, 6, 10, 17, 18, 15, 12, 1]
  Label IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 12, 1]
  Lengths: char=12, label=12

=== Dataset Statistics ===
Total samples: 2369250
Average char sequence length: 13.43
Average char sequence length: 13.43
Max char sequence length: 17
Min char sequence length: 6
Max char sequence length: 17
Min char sequence length: 6


In [14]:

# Debug: Verify label structure in detail
print("=== Label Structure Deep Dive ===\n")

# Analyze multiple samples
for sample_idx in range(min(5, len(df_train))):
    row = df_train.iloc[sample_idx]
    src = row['src']
    tgt = row['tgt']
    char_ids = row['char_id_list']
    label_ids = row['label_id_list']
    
    eq_pos = src.index('=')
    
    print(f"Sample {sample_idx + 1}:")
    print(f"  Source: {src}")
    print(f"  Target: {tgt}")
    print(f"  '=' position in src: {eq_pos}")
    print(f"  Source length: {len(src)}")
    print(f"  Label length: {len(label_ids)}")
    
    # Check label structure
    pads_before_answer = sum(1 for i in range(eq_pos + 1) if label_ids[i] == 0)
    answer_start_idx = eq_pos + 1
    answer_part = label_ids[answer_start_idx:-1]  # Exclude <eos>
    
    print(f"  Label structure:")
    print(f"    - Pad tokens (0-{eq_pos}): {pads_before_answer} positions should be pad")
    print(f"    - Answer starts at position: {answer_start_idx}")
    print(f"    - Answer IDs: {answer_part}")
    print(f"    - Answer decoded: {''.join([id_to_char[id] for id in answer_part])}")
    print(f"    - <eos> position: {len(label_ids) - 1}")
    
    # Verify
    all_pads = all(label_ids[i] == 0 for i in range(eq_pos + 1))
    print(f"  Status: {'✓ Correct' if all_pads else '✗ Issue: Not all pads before answer'}")
    print()


=== Label Structure Deep Dive ===

Sample 1:
  Source: 14*(43+20)=882
  Target: 882
  '=' position in src: 10
  Source length: 14
  Label length: 15
  Label structure:
    - Pad tokens (0-10): 11 positions should be pad
    - Answer starts at position: 11
    - Answer IDs: [16, 16, 10]
    - Answer decoded: 882
    - <eos> position: 14
  Status: ✓ Correct

Sample 2:
  Source: (6+1)*5=35
  Target: 35
  '=' position in src: 7
  Source length: 10
  Label length: 11
  Label structure:
    - Pad tokens (0-7): 8 positions should be pad
    - Answer starts at position: 8
    - Answer IDs: [11, 13]
    - Answer decoded: 35
    - <eos> position: 10
  Status: ✓ Correct

Sample 3:
  Source: 13+32+29=74
  Target: 74
  '=' position in src: 8
  Source length: 11
  Label length: 12
  Label structure:
    - Pad tokens (0-8): 9 positions should be pad
    - Answer starts at position: 9
    - Answer IDs: [15, 12]
    - Answer decoded: 74
    - <eos> position: 11
  Status: ✓ Correct

Sample 4:
  Source: 

# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [9]:
batch_size = 64
epochs = 2
embed_dim = 256
hidden_dim = 256
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [10]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    
    def __len__(self):
        # return the amount of data
        return len(self.sequences)
    
    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        row = self.sequences.iloc[index]
        x = row['char_id_list']
        y = row['label_id_list']
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])
    
    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens


In [11]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [32]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train, 
                                       batch_size=batch_size,
                                       shuffle=True,
                                       collate_fn=collate_fn,
                                       num_workers=0)

# Test the dataloader
print("=== DataLoader Test ===\n")
print(f"Total batches in training set: {len(dl_train)}")

# Get one batch to verify the shape
batch_x, batch_y, batch_x_lens, batch_y_lens = next(iter(dl_train))

print(f"\nFirst batch shapes:")
print(f"  batch_x shape: {batch_x.shape} (batch_size, seq_length)")
print(f"  batch_y shape: {batch_y.shape} (batch_size, seq_length)")
print(f"  batch_x_lens shape: {batch_x_lens.shape}")
print(f"  batch_y_lens shape: {batch_y_lens.shape}")

print(f"\nBatch details:")
print(f"  Batch size: {batch_x.shape[0]}")
print(f"  Max sequence length in batch: {batch_x.shape[1]}")
print(f"  Sequence lengths: {batch_x_lens[:5].tolist()} ... (showing first 5)")

# Show a sample from the batch
print(f"\nSample from batch (index 0):")
print(f"  Input IDs: {batch_x[0].tolist()}")
print(f"  Label IDs: {batch_y[0].tolist()}")
print(f"  Input length: {batch_x_lens[0].item()}")
print(f"  Label length: {batch_y_lens[0].item()}")

# Convert back to characters to verify
sample_input = [id_to_char[int(id)] for id in batch_x[0][:batch_x_lens[0]]]
sample_label = [id_to_char[int(id)] for id in batch_y[0][:batch_y_lens[0]]]
print(f"\nSample decoded:")
print(f"  Input: {''.join(sample_input)}")
print(f"  Label: {''.join(sample_label)}")



=== DataLoader Test ===

Total batches in training set: 37020

First batch shapes:
  batch_x shape: torch.Size([64, 17]) (batch_size, seq_length)
  batch_y shape: torch.Size([64, 17]) (batch_size, seq_length)
  batch_x_lens shape: torch.Size([64])
  batch_y_lens shape: torch.Size([64])

Batch details:
  Batch size: 64
  Max sequence length in batch: 17
  Sequence lengths: [14, 16, 13, 10, 13] ... (showing first 5)

Sample from batch (index 0):
  Input IDs: [10, 13, 5, 9, 14, 5, 9, 15, 18, 14, 16, 8, 8, 1, 0, 0, 0]
  Label IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 14, 16, 8, 8, 1, 0, 0, 0]
  Input length: 14
  Label length: 14

Sample decoded:
  Input: 25*16*17=6800<eos>
  Label: <pad><pad><pad><pad><pad><pad><pad><pad><pad>6800<eos>


# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [31]:
torch.manual_seed(2)

# Specify a device (cuda or cpu）
# For NVIDIA GPU, use 'cuda'
# For CPU only, use 'cpu'

if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Using NVIDIA CUDA GPU")
else:
    device = torch.device('cpu')
    print("Using CPU (no GPU available)")

print(f"Device: {device}")

# Test if MPS works properly
try:
    # Simple test tensor on MPS
    test_tensor = torch.randn(2, 3).to(device)
    print(f"Device test passed: tensor shape {test_tensor.shape}")
except Exception as e:
    print(f"Device test failed: {e}")
    print("  Falling back to CPU")
    device = torch.device('cpu')


Using CPU (no GPU available)
Device: cpu
Device test passed: tensor shape torch.Size([2, 3])


In [33]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])
        
        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)
    
    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)
        
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        
        batch_x = self.linear(batch_x)
        
        return batch_x
    
    def generator(self, start_char, max_len=200):
        """
        Generate sequence character by character until <eos> or max_len.
        
        Args:
            start_char: Starting string (e.g., "1+2=")
            max_len: Maximum sequence length
            
        Returns:
            List of characters in the generated sequence
        """
        self.eval()  # Set to evaluation mode
        
        # Convert starting characters to IDs
        char_list = [char_to_id.get(c, char_to_id['<unk>']) for c in start_char]
        
        with torch.no_grad():  # No gradient computation needed
            while len(char_list) < max_len:
                # 1. Pack the char_list to tensor
                x_tensor = torch.LongTensor([char_list]).to(device)  # (1, seq_len)
                x_lens = torch.LongTensor([len(char_list)]).to(device)  # (1,)
                
                # 2. Input to the encoder (embedding -> LSTM -> linear)
                y = self.encoder(x_tensor, x_lens)  # (1, seq_len, vocab_size)
                
                # 3. Obtain the next token prediction (last time step)
                # Get the output for the last position in the sequence
                last_output = y[0, -1, :]  # (vocab_size,)
                
                # 4. Use argmax function to get the next token prediction
                next_char = torch.argmax(last_output).item()
                
                # 5. Check if we generated <eos>
                if next_char == char_to_id['<eos>']:
                    char_list.append(next_char)
                    break
                
                # 6. Append the predicted character and continue
                char_list.append(next_char)
        
        # Convert IDs back to characters
        return [id_to_char.get(ch_id, '<unk>') for ch_id in char_list]


In [34]:
# Create the model
model = CharRNN(vocab_size, embed_dim, hidden_dim)

print(f"\nModel created successfully!")
print(f"Model summary:")
print(f"  - Vocab size: {vocab_size}")
print(f"  - Embedding dim: {embed_dim}")
print(f"  - Hidden dim: {hidden_dim}")
print(f"  - Device: {device}")



Model created successfully!
Model summary:
  - Vocab size: 19
  - Embedding dim: 256
  - Hidden dim: 256
  - Device: cpu


# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [35]:
# Cross-entropy loss function. The loss function should ignore <pad>
criterion = torch.nn.CrossEntropyLoss(
    ignore_index=char_to_id['<pad>'],
    reduction='mean'
)

# Use Adam optimizer for parameter updates
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print("Loss and Optimizer initialized:")
print(f"  - Loss function: CrossEntropyLoss (ignoring <pad>)")
print(f"  - Optimizer: Adam")
print(f"  - Learning rate: {lr}")
print(f"  - Gradient clip: {grad_clip}")


Loss and Optimizer initialized:
  - Loss function: CrossEntropyLoss (ignoring <pad>)
  - Optimizer: Adam
  - Learning rate: 0.001
  - Gradient clip: 1


In [30]:
from tqdm import tqdm

model = model.to(device)
model.train()

print("="*60)
print("Starting Training")
print("="*60)

i = 0
train_losses = []

for epoch in range(1, epochs+1):
    epoch_losses = []
    
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Clear the gradient
        optimizer.zero_grad()
        
        # Move data to device
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_x_lens = batch_x_lens.to(device)
        
        # Forward pass: get model predictions
        batch_pred_y = model(batch_x, batch_x_lens)
        
        # Reshape for loss calculation
        # batch_pred_y: (batch_size, seq_len, vocab_size)
        # batch_y: (batch_size, seq_len)
        batch_pred_y_reshaped = batch_pred_y.reshape(-1, vocab_size)
        batch_y_reshaped = batch_y.reshape(-1)
        
        # Calculate loss (ignore <pad> tokens)
        loss = criterion(batch_pred_y_reshaped, batch_y_reshaped)
        
        # Back propagation
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip)
        
        # Optimize parameters
        optimizer.step()
        
        # Track loss
        epoch_losses.append(loss.item())
        train_losses.append(loss.item())
        
        i += 1
        if i % 50 == 0:
            avg_loss = np.mean(epoch_losses[-50:])
            bar.set_postfix(loss=f"{avg_loss:.4f}")
    
    # Epoch summary
    epoch_loss = np.mean(epoch_losses)
    print(f"\nEpoch {epoch} - Average Loss: {epoch_loss:.4f}")
    
    # Evaluate your model
    print(f"\nRunning validation on epoch {epoch}...")
    model.eval()
    matched = 0
    total = 0
    
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}", total=len(df_eval))
    
    with torch.no_grad():
        for _, row in bar_eval:
            batch_x = row['src']
            batch_y = row['tgt']
            
            # Generate prediction using the generator method
            try:
                # Find the position of '=' in the source
                eq_idx = batch_x.index('=')
                # Use the part up to and including '=' as the start
                prediction_str = ''.join(model.generator(batch_x[:eq_idx+1]))
                
                # Extract just the answer part (after '=')
                if '=' in prediction_str:
                    predicted_answer = prediction_str.split('=')[-1].replace('<eos>', '').strip()
                else:
                    predicted_answer = prediction_str.replace('<eos>', '').strip()
                
                # Compare with ground truth
                if predicted_answer == str(batch_y):
                    matched += 1
            except Exception as e:
                # Skip if prediction fails
                pass
            
            total += 1
            bar_eval.update(1)
    
    # Compute exact match (EM) on the eval dataset
    em = matched / total if total > 0 else 0
    print(f"Validation EM: {em:.4f} ({matched}/{total})")
    print(f"{"="*60}\n")
    
    model.train()

print("="*60)
print("Training Complete!")
print("="*60)
print(f"Final training loss: {train_losses[-1]:.4f}")
print(f"Average loss over all epochs: {np.mean(train_losses):.4f}")


Starting Training


Train epoch 1:   0%|          | 0/37020 [00:00<?, ?it/s]



RuntimeError: 'lengths' argument should be a 1D CPU int64 tensor, but got 1D mps:0 Long tensor